In [1]:
from pathlib import Path

import polars as pl
import duckdb

In [2]:
silver_dir = Path("../data/silver")

publications_files = sorted(silver_dir.glob("silver_publications_*.parquet"))
authors_files = sorted(silver_dir.glob("silver_publication_authors_*.parquet"))
quality_files = sorted(silver_dir.glob("silver_quality_issues_*.parquet"))
validation_files = sorted(silver_dir.glob("silver_validation_report_*.parquet"))

publications_path = publications_files[-1]
authors_path = authors_files[-1]
quality_path = quality_files[-1]
validation_path = validation_files[-1]

publications_path, authors_path, quality_path, validation_path

(WindowsPath('../data/silver/silver_publications_20260622T205641.parquet'),
 WindowsPath('../data/silver/silver_publication_authors_20260622T205641.parquet'),
 WindowsPath('../data/silver/silver_quality_issues_20260622T205641.parquet'),
 WindowsPath('../data/silver/silver_validation_report_20260622T205641.parquet'))

In [3]:
sql_path = Path("../analytics_sql/01_author_publication_stats.sql")

query = sql_path.read_text().format(
    authors_path=authors_path.as_posix()
)

duckdb.sql(query)

┌──────────────────────────┬───────────────────┬────────────────────┐
│       author_name        │ publication_count │ first_author_count │
│         varchar          │       int64       │       int128       │
├──────────────────────────┼───────────────────┼────────────────────┤
│ Dimitrios Belias         │                 3 │                  3 │
│ Apostolos Giovanis       │                 3 │                  2 │
│ Alexandros Sahinidis     │                 3 │                  2 │
│ Christos Skourlas        │                 3 │                  1 │
│ Athanasios Koustelios    │                 3 │                  0 │
│ Konstantinos Varsanis    │                 3 │                  0 │
│ Vasiliki Amarantou       │                 2 │                  2 │
│ George Stalidis          │                 2 │                  2 │
│ Lambros Tsourgiannis     │                 2 │                  2 │
│ Petros Belsis            │                 2 │                  1 │
│ Manolis Chalaris  

In [5]:
query = Path("../analytics_sql/02_publisher_type_summary.sql").read_text().format(
    publications_path=publications_path.as_posix()
)

duckdb.sql(query).df()

,publisher,publication_type,publication_count,avg_citation_count
0,i-das,journal-article,74,1.405405
1,Elsevier BV,journal-article,20,2.050000
2,"Human Kinetics, Champaign",other,17,0.000000
3,Museo Argentino de Ciencia Naturales,journal-article,10,0.000000
4,Maad Rayan Publishing Company,journal-article,8,1.875000
5,Biodiversity Heritage Library,journal-article,8,0.000000
6,Longwoods Publishing,journal-article,8,3.875000
7,Negah Scientific Publisher,journal-article,8,3.000000
8,JMIR Publications Inc.,journal-article,5,5.400000
9,Instituto Europeu de Ciencias da Cultura Padre...,journal-article,4,0.000000


In [32]:
query = Path("../analytics_sql/03_quality_issue_summary.sql").read_text().format(
    quality_path=quality_path.as_posix()
)

duckdb.sql(query).df()

,issue_type,affected_publications,affected_rows
0,future_published_date,200,200.0
1,author_name_only_unparsed,30,95.0
2,missing_authors,24,24.0
3,missing_title,4,4.0
